# EN Transformer NER Training on Colab GPU

**重要**: ランタイム → ランタイムのタイプを変更 → **T4 GPU** → 保存 してから実行

- `spacy[cuda12x]` は使わない（Colabのcupyを壊す）
- `spacy init config --gpu` で公式GPU configを生成
- `pip install spacy spacy-transformers` のみ

## 結果 (test set F1=96.56%)
| Entity | F1 | CNN比 |
|---|---|---|
| Overall | 96.56% | +1.3pt |
| PERSON | 96.18% | +0.4 |
| ORGANIZATION | 95.71% | +1.4 |
| ADDRESS | 99.20% | +1.2 |
| BANK_ACCOUNT | 97.64% | +6.1 |
| DATE_OF_BIRTH | 92.49% | +0.8 |

In [ ]:
# Step 1: Install & verify GPU
!pip install -q spacy spacy-transformers
!nvidia-smi
import spacy, thinc.util
print(f'spacy={spacy.__version__}, cupy={thinc.util.has_cupy}')
assert thinc.util.has_cupy, 'CuPy not available! Check GPU runtime.'

In [ ]:
# Step 2: Clone repo & generate GPU config
!git clone https://github.com/plenoai/pleno-anonymize.git 2>&1 | tail -3
%cd /content/pleno-anonymize/packages/training
!python -m spacy init config --lang en --pipeline transformer,ner --gpu /tmp/base.cfg
!python -m spacy init fill-config /tmp/base.cfg configs/gpu.cfg

In [ ]:
# Step 3: Fetch training data & train
!git fetch origin tmp/en-data 2>&1 | tail -1
!git checkout origin/tmp/en-data -- data/processed/en/
!ls data/processed/en/
!python -m spacy train configs/gpu.cfg \
    --output output/en-transformer \
    --paths.train data/processed/en/train.spacy \
    --paths.dev data/processed/en/dev.spacy \
    --gpu-id 0

In [ ]:
# Step 4: Evaluate on test set
!python -m spacy evaluate output/en-transformer/model-best data/processed/en/test.spacy --gpu-id 0 --output output/en-transformer/test_scores.json
!cat output/en-transformer/test_scores.json | python -m json.tool

In [ ]:
# Step 5: Save model to Google Drive (persistent storage)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/pleno-models
!cp -r output/en-transformer/model-best /content/drive/MyDrive/pleno-models/en_transformer_best
!python -m spacy package output/en-transformer/model-best /content/drive/MyDrive/pleno-models/ --name ner_en_transformer --version 0.1.0
print('Model saved to Google Drive!')